In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import argparse
from pathlib import Path
from argparse import Namespace

from sklearn.model_selection import train_test_split

#torch.set_float32_matmul_precision('high')  # allows TF32
#torch.backends.cuda.matmul.allow_tf32 = True
#torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from train_sm import MakeDataset, create_data_loaders, EarlyStopping
from sm import NeuralNet

args = Namespace(
    data_dir=Path(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/data_vMB"),
    num_batches=200,
    batch_size=256,
    x_train_mean=None,
    x_train_std=None,
    y_train_mean=None,
    y_train_std=None,
    layer_dims=[7, 90, 90, 90, 90, 1],
    num_epochs=500,
    lr=1e-3,
    save_name=Path(f"/home/hd/hd_hd/hd_gy283/kmc_project/models/sm_vMB_1e7"),
)

train_loader, test_loader, x_train_mean, x_train_std, y_train_mean, y_train_std = create_data_loaders(args.data_dir, args.num_batches, args.batch_size)
args.x_mean = x_train_mean
args.x_std = x_train_std
args.y_mean = y_train_mean
args.y_std = y_train_std
print(args)

cuda
raw_inputs shape: (200000, 8)
outputs shape: (200000,)
inputs shape: (200000, 7)
Namespace(data_dir=PosixPath('/gpfs/bwfor/work/ws/hd_gy283-my_data/data_vMB'), num_batches=200, batch_size=512, x_train_mean=None, x_train_std=None, y_train_mean=None, y_train_std=None, layer_dims=[7, 90, 90, 90, 90, 1], num_epochs=500, lr=0.001, save_name=PosixPath('/home/hd/hd_hd/hd_gy283/kmc_project/models/sm_vMB_1e7'), x_mean=array([ 1.05583269e-03, -3.87125187e-04, -2.74953601e-05,  6.00693143e-04,
        5.73941391e-05, -1.15724918e-03, -4.64607284e-04]), x_std=array([0.86498467, 0.86631018, 0.86628095, 0.86568466, 0.86639411,
       0.86553085, 0.86658089]), y_mean=-0.03441541679787778, y_std=0.14735276438969477)


In [10]:
model = NeuralNet(args.layer_dims, args.x_mean, args.x_std, args.y_mean, args.y_std).to(device)
#model = torch.compile(model)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=200, gamma=0.1)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=2, eta_min=1e-6)

train_losses = []
val_losses = []

early_stopping = EarlyStopping(patience=10, min_delta=0.0001)

for epoch in range(1, args.num_epochs+1):

    model.train()
    running_loss = 0.0
    for inputs_batch, targets_batch in train_loader:
        inputs_batch  = inputs_batch.to(device, non_blocking=True).float()
        targets_batch = targets_batch.to(device, non_blocking=True).float()

        optimizer.zero_grad()
        preds = model(inputs_batch)
        preds = (preds - model.y_mean) / model.y_std
        targets_batch = (targets_batch - model.y_mean) / model.y_std
        loss  = criterion(preds, targets_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs_batch.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs_batch, targets_batch in test_loader:
            inputs_batch  = inputs_batch.to(device, non_blocking=True).float()
            targets_batch = targets_batch.to(device, non_blocking=True).float()

            preds = model(inputs_batch)
            preds = (preds - model.y_mean) / model.y_std
            targets_batch = (targets_batch - model.y_mean) / model.y_std
            loss  = criterion(preds, targets_batch)
            val_loss += loss.item() * inputs_batch.size(0)

    epoch_val_loss = val_loss / len(test_loader.dataset)
    #scheduler.step()

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    #current_lr = optimizer.param_groups[0]['lr']
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d}/{args.num_epochs} "
              f"   Train Loss: {epoch_train_loss:.10f}"
              f"   Val Loss: {epoch_val_loss:.10f}", flush=True)
    #early_stopping(epoch_val_loss)
    if early_stopping.early_stop:
        print("[EARLY STOPPING TRIGGERED]")
        break

torch.save({
    "model_state_dict": model.state_dict(),
    "args": vars(args),
    "train_losses": train_losses,
    "val_losses": val_losses
}, f"{args.save_name}.pth")

print(f"Model has been saved!")

Epoch 10/500    Train Loss: 0.0044678290   Val Loss: 0.0040161473
Epoch 20/500    Train Loss: 0.0027668395   Val Loss: 0.0028805072
Epoch 30/500    Train Loss: 0.0021567318   Val Loss: 0.0021738989
Epoch 40/500    Train Loss: 0.0017820470   Val Loss: 0.0020582205
Epoch 50/500    Train Loss: 0.0016519549   Val Loss: 0.0016239903
Epoch 60/500    Train Loss: 0.0013982904   Val Loss: 0.0016066025
Epoch 70/500    Train Loss: 0.0012745290   Val Loss: 0.0014676734
Epoch 80/500    Train Loss: 0.0011638462   Val Loss: 0.0013608844
Epoch 90/500    Train Loss: 0.0011064814   Val Loss: 0.0012862002
Epoch 100/500    Train Loss: 0.0010426258   Val Loss: 0.0011165543
Epoch 110/500    Train Loss: 0.0009616060   Val Loss: 0.0011430957
Epoch 120/500    Train Loss: 0.0009625186   Val Loss: 0.0014420408
Epoch 130/500    Train Loss: 0.0009117032   Val Loss: 0.0010986135
Epoch 140/500    Train Loss: 0.0008580052   Val Loss: 0.0012600837
Epoch 150/500    Train Loss: 0.0008484453   Val Loss: 0.0010592818
Epoc